In [2]:
import os
import json
import asyncio
from typing import TypedDict, List, Dict
from langgraph.graph import StateGraph, END
from pydantic import BaseModel, Field
from dotenv import load_dotenv
load_dotenv()
# AutoGen Native Chat imports
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo , SystemMessage, UserMessage




In [2]:
api_key = os.environ.get("GROQ_API_KEY")
if not api_key:
        print("❌ Missing GROQ_API_KEY. Defaulting to empty graph ingestion.")

In [3]:
groq_llm = OpenAIChatCompletionClient(
    model="llama-3.3-70b-versatile",
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",
    extra_create_params={
        "response_format": {
            "type": "json_object"
        }
    },
    model_info=ModelInfo(
        vision=False,
        function_calling=True,
        json_output=True,
        family="unknown"
    )
)

c:\Users\Rehan\.vscode\AGI\.venv\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


In [4]:
import nest_asyncio

# Re-engineers Python's internal loop tracker to allow nesting
nest_asyncio.apply() 


In [5]:

async def main():
    # 1. Grab your API key from environment variables (recommended)
    api_key = os.environ.get("GROQ_API_KEY", "your_fallback_groq_api_key_here")

    # 2. Your exact configuration snippet
    groq_llm = OpenAIChatCompletionClient(
        model="llama-3.3-70b-versatile",
        api_key=api_key,
        base_url="https://api.groq.com/openai/v1",
        extra_create_params={
            "response_format": { "type": "json_object" }
        },
        model_info=ModelInfo(
            vision=False,
            function_calling=True,
            json_output=True,
            family="unknown"
        )
    )

    # 3. Test the output with a prompt that forces a JSON structure
    print("Sending request to Groq...")
    response = await groq_llm.create(
        messages=[
            SystemMessage(content="You are a helpful assistant. You must always output valid JSON."),
            UserMessage(content="Give me a JSON object listing three colors and their hex codes.", source="user")
        ]
    )

    # 4. Print the raw string response
    print("\n--- Model Response ---")
    print(response.content)

# Run the async main function
asyncio.run(main())


c:\Users\Rehan\.vscode\AGI\.venv\Lib\site-packages\autogen_ext\models\openai\_openai_client.py:466: UserWarning: Missing required field 'structured_output' in ModelInfo. This field will be required in a future version of AutoGen.
  validate_model_info(self._model_info)


Sending request to Groq...


NotFoundError: Error code: 404 - {'error': {'message': 'The model `llama-3.3-70b-versatile` does not exist or you do not have access to it.', 'type': 'invalid_request_error', 'code': 'model_not_found'}}

In [6]:
from groq import Groq
import os

client = Groq(api_key=os.environ["GROQ_API_KEY"])

response = client.chat.completions.create(
    model="qwen/qwen3.6-27b",

    messages=[
        {
            "role": "user",
            "content": """
Return ONLY valid JSON.

Do not provide reasoning.
Do not provide explanations.
Do not use <think> tags.

Return exactly:
{"hello":"world"}
"""
        }
    ],

    reasoning_effort="none",
    reasoning_format="hidden",

    response_format={
        "type": "json_object"
    }
)

print("CONTENT:")
print(response.choices[0].message.content)

print("\nFULL MESSAGE:")
print(response.choices[0].message)

CONTENT:
{"hello":"world"}

FULL MESSAGE:
ChatCompletionMessage(content='{"hello":"world"}', role='assistant', annotations=None, executed_tools=None, function_call=None, reasoning=None, tool_calls=None)


In [4]:
import os
from autogen_agentchat.agents import AssistantAgent
from autogen_ext.models.openai import OpenAIChatCompletionClient
from autogen_core.models import ModelInfo

api_key = os.environ["GROQ_API_KEY"]
model_client = OpenAIChatCompletionClient(
    model="qwen/qwen3.6-27b",
    api_key=api_key,
    base_url="https://api.groq.com/openai/v1",

    reasoning_effort="none",

    response_format={
        "type": "json_object"
    },

    model_info=ModelInfo(
        vision=False,
        function_calling=True,
        json_output=True,
        family="unknown"
    )
)

agent = AssistantAgent(
    name="QwenTest",
    model_client=model_client,
    system_message=(
        "Return only valid JSON. "
        "Do not provide reasoning or explanations."
    )
)

result = await agent.run(
    task='Return a JSON object containing the answer to 2 + 2. '
         'Use this format: {"answer": 4}'
)

print(result.messages[-1].content)
print(f"print(model_client._create_args) : {model_client._create_args}")
print(f"result.messages[-1].content: {result.messages[-1].content}")


{"answer": 4}
print(model_client._create_args) : {'model': 'qwen/qwen3.6-27b', 'reasoning_effort': 'none', 'response_format': {'type': 'json_object'}}
result.messages[-1].content: {"answer": 4}


In [8]:
import autogen_agentchat
import autogen_ext

print("autogen-agentchat:", getattr(autogen_agentchat, "__version__", "unknown"))
print("autogen-ext:", getattr(autogen_ext, "__version__", "unknown"))

autogen-agentchat: 0.7.5
autogen-ext: 0.7.5


In [9]:
import importlib.metadata

print("autogen-agentchat:", importlib.metadata.version("autogen-agentchat"))
print("autogen-ext:", importlib.metadata.version("autogen-ext"))

autogen-agentchat: 0.7.5
autogen-ext: 0.7.5


In [10]:
print(model_client._create_args)

{'model': 'qwen/qwen3.6-27b'}


In [11]:
import inspect

print(inspect.signature(OpenAIChatCompletionClient))

(**kwargs: Unpack[autogen_ext.models.openai.config.OpenAIClientConfiguration])


In [12]:
from autogen_ext.models.openai.config import OpenAIClientConfiguration
from typing import get_type_hints

print(get_type_hints(OpenAIClientConfiguration))

{'frequency_penalty': typing.Optional[float], 'logit_bias': typing.Optional[typing.Dict[str, int]], 'max_tokens': typing.Optional[int], 'n': typing.Optional[int], 'presence_penalty': typing.Optional[float], 'response_format': <class 'autogen_ext.models.openai.config.ResponseFormat'>, 'seed': typing.Optional[int], 'stop': typing.Union[str, NoneType, typing.List[str]], 'temperature': typing.Optional[float], 'top_p': typing.Optional[float], 'user': <class 'str'>, 'stream_options': typing.Optional[autogen_ext.models.openai.config.StreamOptions], 'parallel_tool_calls': typing.Optional[bool], 'reasoning_effort': typing.Optional[typing.Literal['minimal', 'low', 'medium', 'high']], 'model': <class 'str'>, 'api_key': <class 'str'>, 'timeout': typing.Optional[float], 'max_retries': <class 'int'>, 'model_capabilities': <class 'autogen_core.models._model_client.ModelCapabilities'>, 'model_info': <class 'autogen_core.models._model_client.ModelInfo'>, 'add_name_prefixes': <class 'bool'>, 'include_na